# Colab runner

Runs the repo's own commands; no analysis logic lives here. Use a GPU runtime (Runtime → Change runtime type → T4 GPU).

Paste the printed outputs back for review.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/<your-username>/UnderwaterAcousticClassification.git'  # edit
!git clone -q $REPO_URL /content/uac || (cd /content/uac && git pull -q)
%cd /content/uac
!pip install -q -e '.[dev]'
!python -m pytest -q tests

In [ ]:
import os
os.environ['UAC_DATA_ROOT'] = '/content/drive/MyDrive/uac_data'
!mkdir -p $UAC_DATA_ROOT

## Development run: public DeepShip subset (not reported)

In [ ]:
!python data/download.py deepship_github --data-root $UAC_DATA_ROOT
!python -m src.run --config configs/deepship_github.yaml 2>&1 | grep -v '^\[cache\]'
!cat results/deepship_github/summary.md

## Main run (after the full datasets have arrived and been placed in `$UAC_DATA_ROOT/raw/`)

See `data/README.md`. Record checksums once, then run everything. The first run builds the feature cache on Drive; later runs reuse it.

In [ ]:
!python data/download.py checksums deepship --data-root $UAC_DATA_ROOT
!python data/download.py checksums shipsear --data-root $UAC_DATA_ROOT

In [ ]:
!python -m src.run --config configs/main.yaml --rebuild-manifests 2>&1 | grep -v '^\[cache\]'
!cat results/main/summary.md

In [ ]:
# Save results to Drive so they survive the session; then commit results/main to the repo.
!cp -r results /content/drive/MyDrive/uac_results